# Early Detection of Alzheimer's Disease via EfficientNetB0–ResNet50 Feature-Fusion Ensemble
## Multi-seed reproducibility run + quantitative Grad-CAM audit (v2, for IEEE Access submission)

**Dataset:** Kaggle *Alzheimer MRI Preprocessed Dataset* (128×128 preprocessed MRI slices; 6,400 images
total; classes: MildDemented (896), ModerateDemented (64), NonDemented (3200), VeryMildDemented (2240)).

**What's new in this version, and why:** an earlier single-run version of this notebook showed a 2.4
percentage-point accuracy swing between two full runs (90.63% vs 88.23%), purely from random weight
initialization and augmentation — not a reliable, citable number for a journal submission. This version:
1. Fixes the train/val/test split once (so results are comparable across seeds), then trains the fusion
   model AND both single-backbone ablation baselines across **5 random seeds**, reporting **mean ±
   standard deviation** for every metric — the standard a peer reviewer will expect.
2. Runs Grad-CAM on **multiple images per class** (not just one) and computes a quantitative
   corner-vs-center activation score, to check whether an earlier single-image finding (the
   VeryMildDemented example's attention falling on image corners rather than brain tissue) is a
   consistent pattern or was specific to that one image.

**Kaggle setup:** Settings → Accelerator → GPU T4 x2 (or P100). Add the dataset via
"Add Input" → search for the class-balance fingerprint 896 / 64 / 3200 / 2240 → Add.

**Estimated runtime:** ~40–45 minutes per seed × 5 seeds ≈ 3.5–4 hours total — well inside a
28-hour GPU budget. This is a single Save & Run All (Commit); let it run start to finish.


In [ ]:
# 1. Environment
import os, sys, json, time, random, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0, ResNet50
import matplotlib.pyplot as plt
import matplotlib.cm as cmap_module
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              f1_score, precision_score, recall_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
# 2. Locate every class folder under /kaggle/input, however the dataset happens to be organized.
CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
IMG_SIZE = (224, 224)   # EfficientNetB0 / ResNet50 expect this; source images are natively 128x128
BATCH_SIZE = 32

def normalize(name):
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")

canonical_by_norm = {normalize(c): c for c in CLASSES}

class_dirs = {}
for root, dirs, files in os.walk("/kaggle/input"):
    norm_root_name = normalize(os.path.basename(root))
    if norm_root_name in canonical_by_norm:
        if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
            canon = canonical_by_norm[norm_root_name]
            class_dirs.setdefault(canon, []).append(root)

for c in CLASSES:
    found = class_dirs.get(c, [])
    print(f"{c}: {len(found)} matching folder(s) -> {found}")

assert all(c in class_dirs for c in CLASSES), (
    "Could not find all 4 class folders under /kaggle/input — check the dataset was added via "
    "'Add Input' and that its folder names contain the class names (e.g. MildDemented / Mild_Demented)."
)


In [ ]:
# 3. Build a single file/label index from every matched class folder.
records = []
for cls, dirs_for_cls in class_dirs.items():
    for cls_dir in dirs_for_cls:
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                records.append({"filepath": os.path.join(cls_dir, fname), "label": cls})

df = pd.DataFrame(records)
df = df.drop_duplicates(subset="filepath").reset_index(drop=True)
print("Total images found:", len(df))
print(df["label"].value_counts())
assert len(df) > 0, "No images found — check the dataset was added via Kaggle's 'Add Input' step."


In [ ]:
# 4. Stratified train / val / test split (70 / 15 / 15) — FIXED once, shared across all seed runs below.
# Keeping the split fixed isolates training-randomness (weight init, augmentation order) as the only
# source of variance between seeds; varying the split too would conflate two different kinds of variance.
SPLIT_SEED = 42
label_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SPLIT_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SPLIT_SEED)

print("Train:", len(train_df), dict(train_df.label.value_counts()))
print("Val:  ", len(val_df),   dict(val_df.label.value_counts()))
print("Test: ", len(test_df),  dict(test_df.label.value_counts()))

train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)


In [ ]:
# 5. Class weights (computed once from the fixed train split; identical across all seed runs)
class_labels = sorted(df["label"].unique())
cw = compute_class_weight(class_weight="balanced", classes=np.array(class_labels), y=train_df["label"])
class_weight_dict_str = dict(zip(class_labels, cw))
class_weight_dict = {label_to_idx[k]: v for k, v in class_weight_dict_str.items()}
print("Class weight (index-keyed):", class_weight_dict)


In [ ]:
# 6. tf.data input pipeline builder (parameterized by seed, so shuffle/augmentation order varies per run)
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label

def make_dataset(frame, training=False, seed=42):
    paths = frame["filepath"].values
    labels = np.array([label_to_idx[l] for l in frame["label"].values])
    labels_oh = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES))
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_oh))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=seed)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [ ]:
# 7. Model builders (fusion model + single-backbone baselines) — identical structure every seed run.
def build_fusion_model(num_classes=4, image_size=(224, 224)):
    inputs = layers.Input(shape=(*image_size, 3))
    eff_input = tf.keras.applications.efficientnet.preprocess_input(inputs)
    eff_base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=eff_input, pooling=None)
    eff_base._name = "efficientnetb0_branch"
    eff_features = layers.GlobalAveragePooling2D(name="eff_gap")(eff_base.output)

    res_input = tf.keras.applications.resnet50.preprocess_input(inputs)
    res_base = ResNet50(include_top=False, weights="imagenet", input_tensor=res_input, pooling=None)
    res_base._name = "resnet50_branch"
    res_features = layers.GlobalAveragePooling2D(name="res_gap")(res_base.output)

    for layer in eff_base.layers:
        layer.trainable = False
    for layer in res_base.layers:
        layer.trainable = False

    fused = layers.Concatenate(name="feature_fusion")([eff_features, res_features])
    x = layers.BatchNormalization()(fused)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="EffResNet_FusionNet")
    return model, eff_base, res_base

def unfreeze_top(base, n_layers):
    for layer in base.layers[-n_layers:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

def build_single_backbone(base_ctor, preprocess_fn, num_classes=4, image_size=(224, 224), name="single"):
    inputs = layers.Input(shape=(*image_size, 3))
    x_in = preprocess_fn(inputs)
    base = base_ctor(include_top=False, weights="imagenet", input_tensor=x_in, pooling="avg")
    for layer in base.layers:
        layer.trainable = False
    x = layers.Dense(256, activation="relu")(base.output)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs, name=name), base


In [ ]:
# 8. Evaluation helper
def evaluate_model(m, test_ds):
    y_true, y_pred, y_prob = [], [], []
    for imgs, labels_oh in test_ds:
        probs = m.predict(imgs, verbose=0)
        y_prob.extend(probs)
        y_pred.extend(np.argmax(probs, axis=1))
        y_true.extend(np.argmax(labels_oh.numpy(), axis=1))
    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
    y_true_oh = tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_oh, y_prob, average="macro", multi_class="ovr")
    except Exception:
        auc_macro = None
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro")),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro")),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted")),
        "auc_macro_ovr": float(auc_macro) if auc_macro is not None else None,
    }, y_true, y_pred, confusion_matrix(y_true, y_pred)


In [ ]:
# 9. One full experiment (fusion model + both ablation baselines) for a single seed.
def run_one_seed(seed, epochs_phase1=25, epochs_phase2=40, epochs_baseline=20):
    tf.keras.backend.clear_session()
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    train_ds = make_dataset(train_df, training=True, seed=seed)
    val_ds = make_dataset(val_df, training=False)
    test_ds = make_dataset(test_df, training=False)

    # --- Fusion model, Phase 1 (frozen) then Phase 2 (fine-tune) ---
    model, eff_base, res_base = build_fusion_model(len(CLASSES), IMG_SIZE)
    model.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc", multi_label=True)])
    cb1 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
    hist1 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_phase1,
                       class_weight=class_weight_dict, callbacks=cb1, verbose=0)

    unfreeze_top(eff_base, 30)
    unfreeze_top(res_base, 20)
    model.compile(optimizer=optimizers.Adam(1e-5), loss="categorical_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc", multi_label=True)])
    cb2 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7)]
    hist2 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_phase2,
                       class_weight=class_weight_dict, callbacks=cb2, verbose=0)

    fusion_metrics, y_true, y_pred, cm = evaluate_model(model, test_ds)

    # --- Ablation baselines, same seed, same split, same class weights ---
    def train_and_eval_baseline(base_ctor, preprocess_fn, name):
        m, base = build_single_backbone(base_ctor, preprocess_fn, len(CLASSES), IMG_SIZE, name)
        m.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
        cbk = [callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)]
        m.fit(train_ds, validation_data=val_ds, epochs=epochs_baseline,
              class_weight=class_weight_dict, callbacks=cbk, verbose=0)
        metrics, _, _, _ = evaluate_model(m, test_ds)
        return metrics

    eff_only = train_and_eval_baseline(EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, "eff_only")
    res_only = train_and_eval_baseline(ResNet50, tf.keras.applications.resnet50.preprocess_input, "res_only")

    result = {
        "seed": seed,
        "fusion_accuracy": fusion_metrics["accuracy"], "fusion_f1_macro": fusion_metrics["f1_macro"],
        "fusion_precision_macro": fusion_metrics["precision_macro"], "fusion_recall_macro": fusion_metrics["recall_macro"],
        "fusion_f1_weighted": fusion_metrics["f1_weighted"], "fusion_auc_macro": fusion_metrics["auc_macro_ovr"],
        "eff_only_accuracy": eff_only["accuracy"], "eff_only_f1_macro": eff_only["f1_macro"],
        "res_only_accuracy": res_only["accuracy"], "res_only_f1_macro": res_only["f1_macro"],
    }
    return result, model, eff_base, hist1, hist2, y_true, y_pred, cm


In [ ]:
# 10. MULTI-SEED LOOP — this is the main run. ~40-45 min/seed x 5 seeds ≈ 3.5-4 hours.
SEEDS = [42, 123, 456, 789, 2024]
all_results = []
REFERENCE = {}   # keeps the first seed's trained model + histories for the figures below

results_csv_path = "multiseed_results.csv"
if os.path.exists(results_csv_path):
    os.remove(results_csv_path)

for i, seed in enumerate(SEEDS):
    print(f"\n===== Seed {seed} ({i+1}/{len(SEEDS)}) =====")
    t0 = time.time()
    result, model, eff_base, hist1, hist2, y_true, y_pred, cm = run_one_seed(seed)
    elapsed = time.time() - t0
    result["wall_clock_sec"] = round(elapsed, 1)
    print(f"Seed {seed} done in {elapsed/60:.1f} min — fusion_accuracy={result['fusion_accuracy']:.4f}")
    all_results.append(result)

    # append immediately so partial progress survives even if a later seed crashes
    pd.DataFrame([result]).to_csv(results_csv_path, mode="a",
                                   header=not os.path.exists(results_csv_path) or i == 0, index=False)

    if i == 0:
        REFERENCE = {"model": model, "eff_base": eff_base, "hist1": hist1, "hist2": hist2,
                      "y_true": y_true, "y_pred": y_pred, "cm": cm, "seed": seed}
        model.save("reference_model_seed{}.keras".format(seed))

results_df = pd.DataFrame(all_results)
results_df.to_csv(results_csv_path, index=False)
print("\nAll seeds complete.")
print(results_df)


In [ ]:
# 11. Aggregate mean ± std across seeds — THIS is the headline result for the manuscript now,
# not any single seed's number.
metric_cols = ["fusion_accuracy", "fusion_f1_macro", "fusion_precision_macro", "fusion_recall_macro",
               "fusion_f1_weighted", "fusion_auc_macro", "eff_only_accuracy", "eff_only_f1_macro",
               "res_only_accuracy", "res_only_f1_macro"]

summary = {}
for col in metric_cols:
    summary[col] = {"mean": float(results_df[col].mean()), "std": float(results_df[col].std())}

with open("multiseed_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Mean ± std across", len(SEEDS), "seeds:")
for col in metric_cols:
    print(f"  {col}: {summary[col]['mean']:.4f} ± {summary[col]['std']:.4f}")


In [ ]:
# 12. Training curves for the reference (first-seed) run
def plot_history(h1, h2, out_prefix="training_curve"):
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    epochs_range = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(epochs_range, acc, label="Train")
    axes[0].plot(epochs_range, val_acc, label="Validation")
    axes[0].axvline(len(h1.history["accuracy"]), color="gray", linestyle="--", label="Fine-tune start")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(epochs_range, loss, label="Train")
    axes[1].plot(epochs_range, val_loss, label="Validation")
    axes[1].axvline(len(h1.history["loss"]), color="gray", linestyle="--", label="Fine-tune start")
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{out_prefix}.png", dpi=200)
    plt.show()

plot_history(REFERENCE["hist1"], REFERENCE["hist2"])


In [ ]:
# 13. Confusion matrix for the reference (first-seed) run
cm = REFERENCE["cm"]
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix — EffResNet FusionNet (seed {REFERENCE['seed']})")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

print(classification_report(REFERENCE["y_true"], REFERENCE["y_pred"], target_names=CLASSES, digits=4))


In [ ]:
# 14. Quantitative multi-image Grad-CAM audit (reference-seed model).
# Instead of one image per class, this samples several correctly-classified images per class, computes
# Grad-CAM for each, and scores each heatmap's corner-vs-center activation — directly testing whether
# an earlier single-image finding (VeryMildDemented attention falling on image corners) is a consistent
# pattern or was specific to one image.

model = REFERENCE["model"]
eff_base = REFERENCE["eff_base"]
LAST_CONV_LAYER = "top_conv"
N_SAMPLES_PER_CLASS = 5

def make_gradcam_heatmap(img_array, full_model, conv_layer_owner, last_conv_layer_name, pred_index=None):
    grad_model = models.Model(
        inputs=full_model.input,
        outputs=[conv_layer_owner.get_layer(last_conv_layer_name).output, full_model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def corner_vs_center_score(heatmap_224):
    # corner patches: 20% x 20% of the image at each of the 4 corners; center: middle 60% x 60%
    h, w = heatmap_224.shape
    cs = int(0.2 * h)
    corners = np.concatenate([
        heatmap_224[:cs, :cs].flatten(), heatmap_224[:cs, -cs:].flatten(),
        heatmap_224[-cs:, :cs].flatten(), heatmap_224[-cs:, -cs:].flatten(),
    ])
    c0, c1 = int(0.2 * h), int(0.8 * h)
    center = heatmap_224[c0:c1, c0:c1].flatten()
    return float(corners.mean()), float(center.mean())

def get_samples_for_class(cls_name, n, prefer_correct=True):
    rows = test_df[test_df["label"] == cls_name]
    cls_idx = label_to_idx[cls_name]
    chosen = []
    for _, row in rows.iterrows():
        if len(chosen) >= n:
            break
        img, _ = load_image(row["filepath"], 0)
        arr = tf.expand_dims(img, 0)
        pred = model.predict(arr, verbose=0)
        if (not prefer_correct) or np.argmax(pred[0]) == cls_idx:
            chosen.append((row["filepath"], arr))
    if len(chosen) < n:
        # top up with any remaining images of this class if too few correct predictions exist
        for _, row in rows.iterrows():
            if len(chosen) >= n:
                break
            if row["filepath"] not in [c[0] for c in chosen]:
                img, _ = load_image(row["filepath"], 0)
                chosen.append((row["filepath"], tf.expand_dims(img, 0)))
    return chosen

audit_rows = []
fig, axes = plt.subplots(len(CLASSES), N_SAMPLES_PER_CLASS + 1, figsize=(3.2 * (N_SAMPLES_PER_CLASS + 1), 3.2 * len(CLASSES)))

for ci, cls_name in enumerate(CLASSES):
    samples = get_samples_for_class(cls_name, N_SAMPLES_PER_CLASS)
    axes[ci, 0].text(0.5, 0.5, cls_name, ha="center", va="center", fontsize=13, weight="bold", wrap=True)
    axes[ci, 0].axis("off")
    for si, (fpath, arr) in enumerate(samples):
        heatmap, pred_idx = make_gradcam_heatmap(arr, model, eff_base, LAST_CONV_LAYER)
        heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], IMG_SIZE).numpy().squeeze()
        corner_score, center_score = corner_vs_center_score(heatmap_resized)

        orig = arr[0].numpy().astype("uint8")
        heatmap_color = cmap_module.jet(heatmap_resized)[..., :3] * 255
        overlay = (0.55 * orig + 0.45 * heatmap_color).astype("uint8")

        ax = axes[ci, si + 1]
        ax.imshow(overlay)
        ax.set_title(f"pred: {CLASSES[pred_idx]}\ncorner={corner_score:.2f} center={center_score:.2f}", fontsize=8)
        ax.axis("off")

        audit_rows.append({
            "true_class": cls_name, "predicted_class": CLASSES[pred_idx], "filepath": fpath,
            "corner_activation": corner_score, "center_activation": center_score,
            "corner_minus_center": corner_score - center_score,
        })
    # blank out any unused columns if fewer than N_SAMPLES_PER_CLASS images were available
    for si in range(len(samples), N_SAMPLES_PER_CLASS):
        axes[ci, si + 1].axis("off")

plt.tight_layout()
plt.savefig("gradcam_multi_audit.png", dpi=180, bbox_inches="tight")
plt.show()

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv("gradcam_corner_scores.csv", index=False)
print(audit_df.groupby("true_class")[["corner_activation", "center_activation", "corner_minus_center"]].agg(["mean", "std"]))


## What to copy into the manuscript (multi-seed version)

- `multiseed_results.csv` / `multiseed_summary.json` → headline Results section: report **mean ± std**
  across the 5 seeds for every metric, not a single run's number.
- `confusion_matrix.png`, `training_curve.png` → figures for the reference (first) seed, clearly labeled
  as illustrative of one representative run, not the aggregate.
- `gradcam_multi_audit.png` + `gradcam_corner_scores.csv` → the quantitative interpretability audit.
  Check the per-class corner_minus_center mean/std: if VeryMildDemented's corner_minus_center is
  consistently higher (across images and with low std) than the other classes, that confirms the earlier
  single-image finding as a real, systematic pattern worth reporting as a limitation. If it's inconsistent
  (high std, similar to other classes) then the original single-image finding was likely an anomaly of
  that one image, and Section 5.4 should be corrected to say so rather than repeating the stronger claim.

Do not paste estimated or single-run numbers into the manuscript — use exactly what these files contain.
